# Exercise 4 — E-Commerce Review Analytics

**Purpose:** Prepare the Flipkart review data, calculate product-level average rating and price, and visualize frequently used terms from the review summaries.

The workflow is split into small functions so each cleaning and analysis step can be checked independently.

In [ ]:
%pip install -q kagglehub pandas matplotlib wordcloud openpyxl

from pathlib import Path
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from IPython.display import display

OUT = Path("../output")
OUT.mkdir(exist_ok=True)


## 1. Download and inspect the Kaggle dataset

The notebook searches the downloaded directory for a CSV file and reports the columns found.

In [ ]:
import kagglehub

dataset_dir = kagglehub.dataset_download(
    "niraliivaghani/flipkart-product-customer-reviews-dataset"
)
csv_candidates = glob.glob(str(Path(dataset_dir) / "**" / "*.csv"), recursive=True)

if not csv_candidates:
    raise FileNotFoundError("The downloaded dataset folder does not contain a CSV file.")

csv_path = next(
    (p for p in csv_candidates if Path(p).name.lower() == "dataset-sa.csv"),
    csv_candidates[0]
)

reviews = pd.read_csv(csv_path)
reviews.columns = reviews.columns.str.strip()
print("Loaded:", csv_path)
print("Dataset size:", reviews.shape)
print("Fields:", reviews.columns.tolist())
display(reviews.head())


## 2. Clean the numeric fields

Rating and price may include text formatting. The helper below extracts numeric values and converts invalid entries to missing values.

In [ ]:
def to_numeric_values(column):
    cleaned = (
        column.astype("string")
        .str.replace(",", "", regex=False)
        .str.extract(r"(-?\d+(?:\.\d+)?)", expand=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")

required_fields = ["product_name", "product_price", "Rate", "Summary"]
not_found = [field for field in required_fields if field not in reviews.columns]
if not_found:
    raise KeyError(f"Expected fields not found: {not_found}")

reviews["product_name"] = reviews["product_name"].astype("string").str.strip()
reviews["Rate"] = to_numeric_values(reviews["Rate"])
reviews["product_price"] = to_numeric_values(reviews["product_price"])
reviews["Summary"] = reviews["Summary"].fillna("").astype(str).str.strip()

clean_reviews = reviews.dropna(
    subset=["product_name", "Rate", "product_price"]
).copy()

clean_reviews = clean_reviews[
    clean_reviews["product_name"].ne("")
    & clean_reviews["Rate"].between(1, 5)
    & clean_reviews["product_price"].ge(0)
]

print("Rows before cleaning:", len(reviews))
print("Rows after cleaning:", len(clean_reviews))


## 3. Create the product-level summary

The required output DataFrame is named `items`. Each product receives an average rating and average price.

In [ ]:
items = (
    clean_reviews
    .groupby("product_name", as_index=False)
    .agg(
        average_rating=("Rate", "mean"),
        average_price=("product_price", "mean"),
        review_count=("Summary", "size")
    )
    .sort_values(["average_rating", "review_count"], ascending=[False, False])
)

items[["average_rating", "average_price"]] = items[
    ["average_rating", "average_price"]
].round(2)

display(items.head(12))
items.to_excel(OUT / "Exercise4_Rajveer_Product_Analytics.xlsx", index=False)


## 4. Build a Word Cloud from the Summary field

In [ ]:
review_text = " ".join(
    clean_reviews.loc[clean_reviews["Summary"].str.len() > 0, "Summary"].tolist()
)
review_text = re.sub(r"\s+", " ", review_text).strip()

if not review_text:
    raise ValueError("No usable text was found in the Summary column.")

cloud = WordCloud(
    width=1400,
    height=700,
    background_color="#fbfbf7",
    stopwords=set(STOPWORDS),
    max_words=180,
    collocations=False,
    colormap="plasma",
    contour_width=1
).generate(review_text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(cloud, interpolation="bilinear")
ax.set_title("Common Terms in Customer Review Summaries", fontsize=16, pad=14)
ax.axis("off")
fig.tight_layout()
fig.savefig(OUT / "Exercise4_Rajveer_Review_WordCloud.png", dpi=170, bbox_inches="tight")
plt.show()


## 5. Optional rating overview and final checks

This chart provides an additional view of the product summary without changing the required Word Cloud output.

In [ ]:
top_products = items.nlargest(10, "average_rating").sort_values("average_rating")
if not top_products.empty:
    ax = top_products.plot(
        x="product_name", y="average_rating", kind="barh",
        figsize=(10, 6), legend=False, title="Ten highest average product ratings"
    )
    ax.set_xlabel("Mean rating")
    ax.set_ylabel("Product")
    plt.tight_layout()
    plt.savefig(OUT / "Exercise4_Top_Product_Ratings.png", dpi=160, bbox_inches="tight")
    plt.show()

print("Clean review rows:", len(clean_reviews))
print("Distinct products:", items["product_name"].nunique())
print("Saved product summary and Word Cloud in:", OUT.resolve())


## Conclusion

The `items` table reports product-wise mean rating and mean price. The Word Cloud is built from the cleaned `Summary` text. An additional bar chart provides a quick view of the highest average ratings.